# 1 Million Sample Experiment

Training 6-PLD, 4-PLD, and 3-PLD models on a massive dataset of 1,000,000 samples to establish true asymptotic performance differences.

In [ ]:
import sys, os, copy, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import r2_score
from IPython.display import display, Markdown

project_root = Path.cwd().parent if not (Path.cwd() / 'src').exists() else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

from simulation import SimulationConfig, seed_everything, generate_dataset, select_inputs

# Configuration
SEED = 42
N_TRAIN = 1000000
N_VAL = 10000
N_TEST = 10000
SNR = 10.0
LEARNING_RATE = 1e-3
BATCH_SIZE = 4096 
EPOCHS = 200
PATIENCE = 20

cfg = SimulationConfig()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Dataset Generation

In [ ]:
print("Generating 1 Million training dataset...")
seed_everything(SEED)
torch.manual_seed(SEED)

train_full, Y_train, _ = generate_dataset(N_TRAIN, cfg, SEED + 10, snr=SNR)
val_full, Y_val, _ = generate_dataset(N_VAL, cfg, SEED + 20, snr=SNR)
test_full, Y_test, _ = generate_dataset(N_TEST, cfg, 999, snr=SNR)
print("Datasets generated.")

## Architecture and Training Functions

In [ ]:
class StandardizedNet(nn.Module):
    def __init__(self, input_dim, width):
        super().__init__()
        layers = [nn.Linear(input_dim, width), nn.ELU()]
        for _ in range(8):
            layers += [nn.Linear(width, width), nn.ELU()]
        layers.append(nn.Linear(width, 1))
        self.backbone = nn.Sequential(*layers)
        for layer in self.backbone:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
                nn.init.zeros_(layer.bias)
                
    def forward(self, x):
        return self.backbone(x)

def train_net(net, x_train, y_train, x_val, y_val, mean, std, name):
    yt = ((y_train - mean) / std).astype('float32')
    yv = ((y_val - mean) / std).astype('float32')
    
    train_dataset = TensorDataset(torch.from_numpy(x_train), torch.from_numpy(yt[:, None]))
    val_dataset = TensorDataset(torch.from_numpy(x_val), torch.from_numpy(yv[:, None]))
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
    
    opt = torch.optim.Adam(net.parameters(), lr=LEARNING_RATE)
    loss_fn = nn.L1Loss()
    
    best = float('inf')
    state = copy.deepcopy(net.state_dict())
    stale = 0
    rows = []
    
    for epoch in range(1, EPOCHS + 1):
        net.train()
        train_loss = 0
        for xb, yb in train_loader:
            opt.zero_grad()
            loss = loss_fn(net(xb.to(device)), yb.to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
            train_loss += loss.item()
            
        net.eval()
        val_loss = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                val_loss += loss_fn(net(xb.to(device)), yb.to(device)).item()
        val_loss /= len(val_loader)
        train_loss /= len(train_loader)
        
        rows.append({'epoch': epoch, 'train_mae': train_loss, 'validation_mae': val_loss})
        if epoch % 5 == 0 or epoch == 1:
            print(f"{name} Epoch {epoch}: Train MAE={train_loss:.4f}, Val MAE={val_loss:.4f}")
        
        if val_loss < best:
            best = val_loss
            state = copy.deepcopy(net.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= PATIENCE:
                print(f'{name} early stopped at epoch {epoch}')
                break
                
    net.load_state_dict(state)
    return pd.DataFrame(rows)

def calc_metrics(y, p):
    e = p - y
    return {
        'MAE': np.mean(abs(e)),
        'RMSE': np.sqrt(np.mean(e**2)),
        'MAPE_percent': np.mean(abs(e) / np.maximum(abs(y), 1e-8)) * 100,
        'R2': r2_score(y, p),
        'Pearson': np.corrcoef(y, p)[0, 1],
        'Bias': np.mean(e),
        'Error_SD': np.std(e, ddof=1)
    }

## Experiment Runner

In [ ]:
def run_experiment(pld_name, input_dim, indices=None):
    print(f"\n--- Running {pld_name} Experiment ---")
    if indices is not None:
        X_train = train_full[:, indices]
        X_val = val_full[:, indices]
        X_test = test_full[:, indices]
    else:
        mode_name = pld_name.lower().replace("-", "_")
        X_train = select_inputs(train_full, cfg, mode_name)
        X_val = select_inputs(val_full, cfg, mode_name)
        X_test = select_inputs(test_full, cfg, mode_name)
        
    X_mean = X_train.mean(0, keepdims=True).astype('float32')
    X_std = X_train.std(0, keepdims=True).astype('float32') + 1e-8
    
    X_train = ((X_train - X_mean) / X_std).astype('float32')
    X_val = ((X_val - X_mean) / X_std).astype('float32')
    X_test = ((X_test - X_mean) / X_std).astype('float32')
    
    CBF_mean, CBF_std = float(Y_train[:,0].mean()), float(Y_train[:,0].std()+1e-8)
    ATT_mean, ATT_std = float(Y_train[:,1].mean()), float(Y_train[:,1].std()+1e-8)
    
    cbf_net = StandardizedNet(input_dim, 50).to(device)
    att_net = StandardizedNet(input_dim, 100).to(device)
    
    print(f"Training {pld_name} CBF...")
    cbf_hist = train_net(cbf_net, X_train, Y_train[:,0], X_val, Y_val[:,0], CBF_mean, CBF_std, f"{pld_name}-CBF")
    
    print(f"Training {pld_name} ATT...")
    att_hist = train_net(att_net, X_train, Y_train[:,1], X_val, Y_val[:,1], ATT_mean, ATT_std, f"{pld_name}-ATT")
    
    cbf_net.eval()
    att_net.eval()
    with torch.no_grad():
        cbf_pred = cbf_net(torch.from_numpy(X_test).to(device)).cpu().squeeze(1).numpy() * CBF_std + CBF_mean
        att_pred = att_net(torch.from_numpy(X_test).to(device)).cpu().squeeze(1).numpy() * ATT_std + ATT_mean
    
    pred = np.column_stack((np.clip(cbf_pred, 0, 100), np.clip(att_pred, 0.5, 3.0)))
    
    metrics = []
    for j, param in enumerate(['CBF', 'ATT']):
        row = {'Model': pld_name, 'Parameter': param}
        row.update(calc_metrics(Y_test[:, j], pred[:, j]))
        metrics.append(row)
        
    out_dir = project_root / 'results' / f'model_{pld_name.lower().replace("-", "_")}_1m'
    out_dir.mkdir(exist_ok=True, parents=True)
    
    torch.save(cbf_net.state_dict(), out_dir / 'cbf_model.pt')
    torch.save(att_net.state_dict(), out_dir / 'att_model.pt')
    cbf_hist.to_csv(out_dir / 'cbf_training_history.csv', index=False)
    att_hist.to_csv(out_dir / 'att_training_history.csv', index=False)
    np.savez(out_dir / 'test_predictions.npz', y_true=Y_test, y_pred=pred)
    pd.DataFrame(metrics).to_csv(out_dir / 'metrics.csv', index=False)
    
    return metrics

## Execute Experiments and Compare

In [ ]:
all_metrics = []

# 6 PLD
metrics_6 = run_experiment("6-PLD", 6)
all_metrics.extend(metrics_6)

# 4 PLD
metrics_4 = run_experiment("4-PLD", 4)
all_metrics.extend(metrics_4)

# 3 PLD
THREE_PLD_INDICES = (0, 1, 3)
metrics_3 = run_experiment("3-PLD", 3, indices=THREE_PLD_INDICES)
all_metrics.extend(metrics_3)

# Save combined metrics
combined_df = pd.DataFrame(all_metrics)
combined_df.to_csv(project_root / 'results' / 'comparison_1m_metrics.csv', index=False)
display(Markdown("### Final 1-Million Sample Comparison"))
display(combined_df.round(5))